In [1]:
import pandas as pd

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
cols = [
    "id", "label", "statement", "subject", "speaker",
    "job", "state", "party", "barely_true", "false",
    "half_true", "mostly_true", "pants_fire", "context"
]

train = pd.read_csv("/content/drive/MyDrive/ExAI/train.tsv", sep="\t", header=None, names=cols)
test  = pd.read_csv("/content/drive/MyDrive/ExAI/test.tsv",  sep="\t", header=None, names=cols)
valid = pd.read_csv("/content/drive/MyDrive/ExAI/valid.tsv", sep="\t", header=None, names=cols)

liar = pd.concat([train, test, valid], ignore_index=True)
print(f"Total LIAR samples: {len(liar)}")
print(liar["label"].value_counts())

Total LIAR samples: 11470
label
half-true      2359
false          2248
mostly-true    2211
barely-true    1891
true           1832
pants-fire      929
Name: count, dtype: int64


In [3]:
# Map 6 LIAR labels to binary
# Real (0): true, mostly-true, half-true
# Fake (1): barely-true, false, pants-fire
label_map = {
    "true":        0,
    "mostly-true": 0,
    "half-true":   0,
    "barely-true": 1,
    "false":       1,
    "pants-fire":  1
}

liar["label"] = liar["label"].map(label_map)
liar = liar.dropna(subset=["label", "statement"])
liar["label"] = liar["label"].astype(int)

# LIAR only has statements, no titles
# Use statement as both title and text to match your schema
liar_clean = pd.DataFrame({
    "title": liar["statement"],
    "text":  liar["statement"],
    "label": liar["label"]
})

print(f"LIAR after cleaning: {len(liar_clean)}")
print(liar_clean["label"].value_counts())

LIAR after cleaning: 11470
label
0    6402
1    5068
Name: count, dtype: int64


In [4]:
combined = pd.read_csv("/content/drive/MyDrive/ExAI/combined.csv")
print(f"Existing combined.csv: {len(combined)}")

merged = pd.concat([combined, liar_clean], ignore_index=True)
merged = merged.dropna(subset=["title", "text"])
merged = merged.drop_duplicates(subset=["title", "text"])

print(f"After merge: {len(merged)}")
print(merged["label"].value_counts())
print(merged["label"].value_counts(normalize=True))

Existing combined.csv: 63121
After merge: 74570
label
0    41184
1    33386
Name: count, dtype: int64
label
0    0.552286
1    0.447714
Name: proportion, dtype: float64


In [5]:
merged.to_csv("/content/drive/MyDrive/ExAI/combined.csv", index=False)
print("Saved updated combined.csv")

Saved updated combined.csv
